# KV Cache

**Import Libraries**

In [ ]:
import numpy as np
import time

**Define activation functions**

In [ ]:
class Activations:
  @staticmethod
  def relu(x):
    return np.maximum(0, x)


  @staticmethod
  def softmax(x):
    x_stable = x - np.max(x, axis=-1, keepdims=True)
    exp_x = np.exp(x_stable)
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

## FeedForward Layer

In [ ]:
class Linear:
  def __init__(self, in_features, out_features, bias=True):
    self.in_features = in_features
    self.out_features = out_features
    self.weight = np.random.rand(out_features, in_features)

    if bias:
      self.bias = np.random.rand(out_features)
    else:
      self.bias = 0

  def forward(self, x):
    '''
    input: (N, in_features)
    output: (N, out_features)
    '''
    x = np.dot(x, self.weight.T) + self.bias
    x = Activations.relu(x)
    return x

In [ ]:
class FeedForward:
  def __init__(self, d_model):
    self.layer1 = Linear(d_model, 2*d_model)
    self.layer2 = Linear(2*d_model, d_model)

  def forward(self, x):
    x = self.layer1.forward(x)
    x = self.layer2.forward(x)

    return x

## MHA

**MHA without KVPress**

In [ ]:
class MultiHeadAttention:
  def __init__(self, d_model, num_heads, seq_length):
    self.d_k = d_model // num_heads
    self.d_v = d_model // num_heads
    self.num_heads = num_heads
    self.seq_length = seq_length

    self.causal = self.create_attn_mask()

    self.W_Qs = [np.random.rand(d_model, self.d_k) for _ in range(num_heads)]
    self.W_Ks = [np.random.rand(d_model, self.d_k) for _ in range(num_heads)]
    self.W_Vs = [np.random.rand(d_model, self.d_v) for _ in range(num_heads)]
    self.W_O = np.random.rand(num_heads*self.d_v, d_model)

  def create_attn_mask(self):
    ''' Create a causal seq_length mask for LookAhead MultiHeadAttenion Block in Decoder.

    Arguments:
      None

    Returns:
      attention mask: (seq_length, seq_length)
    '''
    if self.seq_length != 0:
      base = np.zeros((self.seq_length, self.seq_length))
    else:
      base = np.zeros((self.seq_length, self.seq_length))
    row, col = base.shape[0], base.shape[1]

    for r in range(row-1):
      for c in range(col):
        if c > r:
          base[r, c] = np.inf
    return base # (seq_length, seq_length)


  def forward(self, input, causal=False):
    '''
    input: (N, seq_length, d_model)
    '''
    attentions = []
    for i in range(self.num_heads):
      Q = np.dot(input, self.W_Qs[i]) # (N, seq_length, d_k)
      K = np.dot(input, self.W_Ks[i]) # (N, seq_length, d_k)
      V = np.dot(input, self.W_Vs[i]) # (N, seq_length, d_v)
      curr_attention = np.matmul(Q, K.transpose(0, 2, 1)) / np.sqrt(self.d_k) # (N, seq_length, seq_length)

      if causal:
        curr_attention += self.causal_attn_mask # (N, seq_length, seq_length)

      curr_attention = Activations.softmax(curr_attention) # (N, seq_length, seq_length)
      curr_attention = np.matmul(curr_attention, V) # (N, seq_length, seq_length) @ (N, seq_length, d_v) => (N, seq_length, d_v)
      attentions.append(curr_attention)

    attentions = np.concatenate(attentions, axis=-1) # (N, seq_length, num_heads*d_v)
    result = np.dot(attentions, self.W_O) # (N, seq_length, num_heads*d_v) @ (num_heads*d_v, d_model) => (N, seq_length, d_model)
    return result #(N, seq_length, d_model)

**MHA with KVPress**

In [ ]:
class KVPressMultiHeadAttention:
  def __init__(self, d_model, num_heads, seq_length, batch_size):
    self.d_k = d_model // num_heads
    self.d_v = d_model // num_heads
    self.num_heads = num_heads
    self.seq_length = seq_length

    self.causal_attn_mask = self.create_attn_mask()

    # originally
    # W_Q has shape of (d_model, d_k)
    # W_K has shape of (d_model, d_k)
    # W_V has shape of (d_model, d_v)

    self.W_Qs = [np.random.rand(d_model, self.d_k) for _ in range(num_heads)]
    self.W_Ks = [np.random.rand(d_model, self.d_k) for _ in range(num_heads)]
    self.W_Vs = [np.random.rand(d_model, self.d_v) for _ in range(num_heads)]
    self.W_O = np.random.rand(num_heads*self.d_v, d_model)


    self.cached_K = np.zeros((self.num_heads, batch_size, self.seq_length, self.d_k)) # (num_heads, N, seq_length, d_k)
    self.cached_V = np.zeros((self.num_heads, batch_size, self.seq_length, self.d_k)) # (num_heads, N, seq_length, d_k)

  def create_attn_mask(self):
    ''' Create a causal seq_length mask for LookAhead MultiHeadAttention Block in Decoder.

    Returns:
      attention mask: (seq_length, seq_length)
    '''
    base = np.zeros((self.seq_length, self.seq_length))
    for r in range(self.seq_length - 1):
      for c in range(r + 1, self.seq_length):
        base[r, c] = np.inf

    return base  # (seq_length, seq_length)


  def forward(self, input, j, causal=False):
    '''
    input: (N, 1, d_model)
    '''
    attentions = []
    for i in range(self.num_heads):
      Q = np.dot(input, self.W_Qs[i]) # (N, 1, d_k)
      K = np.dot(input, self.W_Ks[i]) # (N, 1, d_k)
      V = np.dot(input, self.W_Vs[i]) # (N, 1, d_v)

      self.cached_K[i, :, j, :] = K[:, 0, :] # (N, 1, d_k)
      self.cached_V[i, :, j, :] = V[:, 0, :] # (N, 1, d_v)

      K = self.cached_K[i, :, :j+1, :] # (N, t, d_k)
      V = self.cached_V[i, :, :j+1, :] # (N, t, d_k)


      curr_attention = np.matmul(Q, K.transpose(0, 2, 1)) / np.sqrt(self.d_k) # (N, 1, 1)

      if causal:
        curr_attention += self.causal_attn_mask[:1, :j+1]

      curr_attention = Activations.softmax(curr_attention) # (N, 1, 1)
      curr_attention = np.matmul(curr_attention, V) # (N, 1, 1) @ (N, 1, d_v) => (N, 1, d_v)
      attentions.append(curr_attention)

    attentions = np.concatenate(attentions, axis=-1) # (N, 1, num_heads*d_v)
    result = np.dot(attentions, self.W_O) # (N, 1, num_heads*d_v) @ (num_heads*d_v, d_model) => (N, 1, d_model)
    return result #(N, 1, d_model)

## Embedding Layer

In [ ]:
class Embedding:
  def __init__(self, vocab_size, embedding_dim):
    self.E = np.random.randn(vocab_size, embedding_dim)

  def forward(self, x):
    '''
    Input:
      x: (N, seq_length)
    Output:
      result: (N, seq_length, embdding_dim)
    '''
    res = []
    for i in range(x.shape[0]):
        curr = self.E[x[i], :] # (seq_length, embedding_dim)
        res.append(curr)
    result = np.stack(res, axis=0)
    return result

## DECODER

In [ ]:
class Decoder_Layer:
  def __init__(self, d_model, num_heads, seq_length, batch_size, kv_press):
    self.kv_press = kv_press

    if kv_press:
      self.mha = KVPressMultiHeadAttention(d_model, num_heads, seq_length, batch_size)
    else:
      self.mha = MultiHeadAttention(d_model, num_heads, seq_length)

    self.ff = FeedForward(d_model)

  def forward(self, x, j):
    if self.kv_press:
      x = self.mha.forward(x, j)
    else:
      x = self.mha.forward(x)

    x = self.ff.forward(x)
    return x

In [ ]:
def softmax(self, x):
  x_stable = x - np.max(x, axis=-1, keepdims=True)
  exp_x = np.exp(x_stable)
  return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

In [ ]:
class Decoder:
  def __init__(self, vocab_size, d_model, num_heads, seq_length, num_layers, kv_press=True):
    self.embedding_layer = Embedding(vocab_size, d_model)
    self.decoder_layers = [Decoder_Layer(d_model, num_heads, seq_length, kv_press) for _ in range(num_layers)]
    self.last_layer = Linear(d_model, vocab_size)

  def forward(self, x):
    x = self.embedding_layer.forward(x)  # (N, T, d_model)
    for decoder_layer in self.decoder_layers:
        x = decoder_layer.forward(x)
    x = self.last_layer.forward(x)
    return x

# Inference

In [ ]:
d_model = 512
num_heads = 8
seq_length = 128
batch_size = 1

**Inference with KV Press**

In [ ]:
x = np.random.rand(1, 1, 512)
full_output = []
decoder_layer = Decoder_Layer(d_model, num_heads, seq_length, batch_size, True)

start_time = time.time()

for j in range(10):
  x = decoder_layer.forward(x, j)
  x = x[:, -1, :]
  x = np.expand_dims(x, axis=1)
  print(x.shape)
  full_output.append(x)

full_output_array = np.stack(full_output, axis=1)
full_output_array = np.squeeze(full_output_array, axis=2)
print("Full Output Shape:", full_output_array.shape)

end_time = time.time()
print(f"Time taken: {end_time - start_time:.4f} seconds")

(1, 1, 512)
(1, 1, 512)
(1, 1, 512)
(1, 1, 512)
(1, 1, 512)
(1, 1, 512)
(1, 1, 512)
(1, 1, 512)
(1, 1, 512)
(1, 1, 512)
Full Output Shape: (1, 10, 512)
Time taken: 0.0514 seconds


**Inference without KVPress**

In [ ]:
decoder_layer = Decoder_Layer(d_model, num_heads, seq_length, batch_size, False)

x = np.random.rand(1, 1, 512)
full_output = [x]
start_time = time.time()

for i in range(1, 10):
    out = decoder_layer.forward(x, i)
    new_token = out[:, -1:, :]  # shape (1, 1, 512)
    x = np.concatenate([x, new_token], axis=1)
    print(x.shape)

end_time = time.time()
print(f"Time taken: {end_time - start_time:.4f} seconds")

(1, 2, 512)
(1, 3, 512)
(1, 4, 512)
(1, 5, 512)
(1, 6, 512)
(1, 7, 512)
(1, 8, 512)
(1, 9, 512)
(1, 10, 512)
Time taken: 0.1290 seconds
